# **SAXS for liquid crystal**

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
from scipy.signal import fftconvolve  # for robust convolution

###############################################################################
# 1. Generic peak + convolution helper functions
###############################################################################

def lorentzian(q, q0, gamma, amplitude=1.0):
    """
    Returns a simple Lorentzian shape centered at q0:
        L(q) = amplitude / [1 + ((q - q0)/gamma)^2].
    """
    return amplitude / (1.0 + ((q - q0)/gamma)**2)

def gaussian_convolution(q, Iq, sigma):
    """
    Convolve Iq with a Gaussian of width sigma using FFT-based convolution.
    This avoids shape-mismatch issues that can occur with np.convolve(..., mode='same').
    """
    if sigma <= 0:
        return Iq
    dq = q[1] - q[0]  # assume evenly spaced q
    n = len(q)
    kernel_halfwidth = n // 2
    q_kernel = np.linspace(-kernel_halfwidth, kernel_halfwidth, 2*kernel_halfwidth + 1) * dq
    gauss_kernel = np.exp(-0.5 * (q_kernel / sigma)**2)
    gauss_kernel /= np.sum(gauss_kernel)
    Iq_conv = fftconvolve(Iq, gauss_kernel, mode='same')
    return Iq_conv

###############################################################################
# 2. Reflection Generators for Hex and Cubic
###############################################################################

def generate_hex_reflections(a, n_peaks):
    """
    Generate the first 'n_peaks' reflections in a 2D hexagonal lattice,
    sorted by the magnitude of q_{h,k}.

    The 2D hex reciprocal spacing is:
      q_{h,k} = (4π / (sqrt(3)*a)) * sqrt(h^2 + h*k + k^2).
    """
    possible_hk = []
    max_index = 15  # search range (adjust as needed)
    for h in range(max_index+1):
        for k in range(max_index+1):
            if h == 0 and k == 0:
                continue
            q_hk = (4.0 * np.pi / (np.sqrt(3.0) * a)) * np.sqrt(h**2 + h*k + k**2)
            possible_hk.append((q_hk, h, k))
    possible_hk.sort(key=lambda x: x[0])
    return possible_hk[:n_peaks]

def generate_cubic_reflections(a, n_peaks):
    """
    Generate the first 'n_peaks' reflections for a naive cubic reciprocal space:
      q_{h,k,l} = (2π / a) * sqrt(h^2 + k^2 + l^2),
    ignoring systematic absences.
    """
    possible_hkl = []
    max_index = 10  # adjust as needed
    for h in range(max_index+1):
        for k in range(max_index+1):
            for l in range(max_index+1):
                if h == 0 and k == 0 and l == 0:
                    continue
                q_hkl = (2.0 * np.pi / a) * np.sqrt(h**2 + k**2 + l**2)
                possible_hkl.append((q_hkl, h, k, l))
    possible_hkl.sort(key=lambda x: x[0])
    return possible_hkl[:n_peaks]

###############################################################################
# 3. Phase-specific scattering models
###############################################################################

def smectic_intensity(q, d, n_peaks, gamma, amplitude,
                      domain_size, resolution, Bfactor=0.0):
    """
    SAXS for a Smectic A phase.
    - Peaks at q_n = n*(2π/d) (n=1...n_peaks)
    - Each peak is multiplied by exp(-Bfactor * q^2)
    - domain_size adds extra broadening ~1/domain_size.
    """
    q0 = 2.0 * np.pi / d
    gamma_eff = gamma + (1.0/domain_size if domain_size > 0 else 0.0)
    Iq = np.zeros_like(q)
    for n in range(1, n_peaks + 1):
        An = amplitude / (n**2)
        lor_val = lorentzian(q, n*q0, gamma_eff, amplitude=An)
        dw_factor = np.exp(-Bfactor * q**2)
        Iq += lor_val * dw_factor
    return gaussian_convolution(q, Iq, resolution)

def nematic_intensity(q, q_peak, gamma, amplitude,
                      domain_size, resolution, Bfactor=0.0):
    """
    SAXS for a Nematic phase: a single broad Lorentzian peak.
    """
    gamma_eff = gamma + (1.0/domain_size if domain_size > 0 else 0.0)
    lor_val = lorentzian(q, q_peak, gamma_eff, amplitude=amplitude)
    dw_factor = np.exp(-Bfactor * q**2)
    Iq = lor_val * dw_factor
    return gaussian_convolution(q, Iq, resolution)

def columnar_hex_intensity(q, a, gamma, amplitude,
                           domain_size, resolution, Bfactor=0.0, n_peaks=3):
    """
    SAXS for a 2D Hexagonal Columnar phase.
    - Sum the first n_peaks reflections from the 2D hex lattice.
    """
    gamma_eff = gamma + (1.0/domain_size if domain_size > 0 else 0.0)
    chosen_hk = generate_hex_reflections(a, n_peaks)
    Iq = np.zeros_like(q)
    for (q_hk, h, k) in chosen_hk:
        lor_val = lorentzian(q, q_hk, gamma_eff, amplitude=amplitude)
        dw_factor = np.exp(-Bfactor * q**2)
        Iq += lor_val * dw_factor
    return gaussian_convolution(q, Iq, resolution)

def bicontinuous_cubic_intensity(q, a, gamma, amplitude,
                                 domain_size, resolution, Bfactor=0.0, n_peaks=4):
    """
    SAXS for a Bicontinuous Cubic phase.
    - Sum the first n_peaks reflections from a naive cubic reciprocal lattice.
    """
    gamma_eff = gamma + (1.0/domain_size if domain_size > 0 else 0.0)
    chosen_hkl = generate_cubic_reflections(a, n_peaks)
    Iq = np.zeros_like(q)
    for (q_hkl, h, k, l) in chosen_hkl:
        lor_val = lorentzian(q, q_hkl, gamma_eff, amplitude=amplitude)
        dw_factor = np.exp(-Bfactor * q**2)
        Iq += lor_val * dw_factor
    return gaussian_convolution(q, Iq, resolution)

###############################################################################
# 4. Dispatcher Function
###############################################################################

def compute_saxs(phase, qmin, qmax, nq,
                 d, q_peak, a, gamma, amplitude,
                 domain_size, resolution, n_peaks,
                 Bfactor):
    """
    Compute the 1D SAXS intensity for a given phase.
    - Smectic: n_peaks controls the number of Bragg orders.
    - Nematic: uses a single peak.
    - Hexagonal Columnar: n_peaks controls the number of reflections.
    - Bicontinuous Cubic: n_peaks controls the number of (h,k,l) reflections.
    """
    q = np.linspace(qmin, qmax, nq)
    if phase == 'Smectic':
        Iq = smectic_intensity(q, d, n_peaks, gamma, amplitude,
                               domain_size, resolution, Bfactor)
    elif phase == 'Nematic':
        Iq = nematic_intensity(q, q_peak, gamma, amplitude,
                               domain_size, resolution, Bfactor)
    elif phase == 'Hexagonal Columnar':
        Iq = columnar_hex_intensity(q, a, gamma, amplitude,
                                    domain_size, resolution, Bfactor, n_peaks)
    elif phase == 'Bicontinuous Cubic':
        Iq = bicontinuous_cubic_intensity(q, a, gamma, amplitude,
                                          domain_size, resolution, Bfactor, n_peaks)
    else:
        Iq = np.zeros_like(q)
    return q, Iq

###############################################################################
# 5. Interactive Widget
###############################################################################

# Define a dropdown for selecting phase
phase_dropdown = Dropdown(
    options=['Smectic', 'Nematic', 'Hexagonal Columnar', 'Bicontinuous Cubic'],
    value='Smectic',
    description='Phase'
)

def interactive_saxs(
    phase='Smectic',
    qmin=0.01, qmax=1.0, nq=500,
    d=50.0,       # layer spacing for Smectic
    q_peak=0.3,   # nematic peak position
    a=50.0,       # lattice parameter for Columnar/Cubic
    gamma=0.02,   # base peak width
    amplitude=1.0,
    domain_size=500.0,
    resolution=0.01,
    n_peaks=10,
    Bfactor=1.0
):
    q, Iq = compute_saxs(
        phase, qmin, qmax, nq,
        d, q_peak, a, gamma, amplitude,
        domain_size, resolution, n_peaks,
        Bfactor
    )
    plt.figure(figsize=(6,4))
    plt.plot(q, Iq, 'b-', lw=2)
    plt.xlabel(r'$q$ [1/nm]')
    plt.ylabel('Intensity (arb. units)')
    plt.title(f'{phase} SAXS Profile')
    plt.yscale('log')
    plt.grid(True)
    plt.show()

interact(
    interactive_saxs,
    phase=phase_dropdown,
    qmin=FloatSlider(value=0.01, min=0.0, max=5.0, step=0.01,
                     continuous_update=False, description='q_min'),
    qmax=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1,
                     continuous_update=False, description='q_max'),
    nq=IntSlider(value=500, min=100, max=2000, step=100,
                 continuous_update=False, description='n_q points'),
    d=FloatSlider(value=50.0, min=5.0, max=200.0, step=5.0,
                  continuous_update=False, description='Smectic d'),
    q_peak=FloatSlider(value=0.3, min=0.1, max=5.0, step=0.01,
                       continuous_update=False, description='Nematic q*'),
    a=FloatSlider(value=50.0, min=5.0, max=200.0, step=5.0,
                  continuous_update=False, description='a (Col/Cubic)'),
    gamma=FloatSlider(value=0.02, min=0.001, max=0.1, step=0.001,
                      continuous_update=False, description='Peak width'),
    amplitude=FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1,
                          continuous_update=False, description='Amplitude'),
    domain_size=FloatSlider(value=500.0, min=10.0, max=1000.0, step=10.0,
                            continuous_update=False, description='Domain Size'),
    resolution=FloatSlider(value=0.01, min=0.0, max=0.1, step=0.01,
                           continuous_update=False, description='Resolution'),
    n_peaks=IntSlider(value=10, min=1, max=20, step=1,
                      continuous_update=False, description='n_peaks'),
    Bfactor=FloatSlider(value=1.0, min=0.0, max=10, step=0.1,
                        continuous_update=False, description='Debye–Waller B')
);

interactive(children=(Dropdown(description='Phase', options=('Smectic', 'Nematic', 'Hexagonal Columnar', 'Bico…